# FlowVision: обучение Donut на новом датасете (52 ТТН) в Google Colab

**Новые данные:** 52 изображения из Label Studio, разметка в `dataset/raw/train1.json`

## Шаг 0: На своём ПК (перед Colab)

```powershell
cd d:\GitHub\FlowVision

# Конвертировать Label Studio → metadata.jsonl + скопировать изображения
python scripts/prepare_data.py

# Упаковать архивы для Colab (включит исправления train/inference)
python scripts/pack_colab.py
```

**Должны появиться в корне проекта:**
- `flowvision_colab_code.zip` (47 KB) — код проекта
- `train_dataset.zip` (23 MB) — 52 изображения + metadata.jsonl

---

## Шаг 1 в Colab: ВКЛЮЧИТЬ GPU

**Среда выполнения → Сменить среду выполнения → T4 GPU (или A100)**

⚠️ Без GPU обучение займет несколько часов на CPU!

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "Включите GPU: Среда выполнения -> Сменить среду выполнения -> T4 GPU"
)
print("GPU:", torch.cuda.get_device_name(0))

## Шаг 2: Загрузить ZIP-файлы и распаковать

Запустите код ниже → появится кнопка **Выбрать файлы** → укажите оба файла:
1. `flowvision_colab_code.zip`
2. `train_dataset.zip`

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

from google.colab import files

# upload пишет в текущую папку — всегда /content (не в FlowVision!)
os.chdir("/content")
print("cwd:", os.getcwd())

print("📁 Шаг 1/2: выберите flowvision_colab_code.zip")
uploaded_code = files.upload()

print("\n📁 Шаг 2/2: выберите train_dataset.zip")
uploaded_data = files.upload()

uploaded = {**uploaded_code, **uploaded_data}

ROOT = Path("/content/FlowVision")
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)

for name in uploaded:
    zip_path = Path("/content") / Path(name).name
    if not zip_path.is_file():
        zip_path = Path(name).resolve()
    if not zip_path.is_file():
        raise FileNotFoundError(f"Не найден после upload: {name}")
    print(f"\n📦 Распаковка: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROOT)
    print("   ✓ OK")

# Проверка структуры
print("\n✓ Проверка структуры...")
meta = ROOT / "dataset" / "train" / "metadata.jsonl"
train_py = ROOT / "train" / "train.py"
app_fmt = ROOT / "app" / "ocr" / "donut_format.py"

assert train_py.exists(), f"❌ Нет {train_py}. Загрузили flowvision_colab_code.zip?"
assert app_fmt.exists(), f"❌ Нет app/ocr/donut_format.py"
assert meta.exists(), f"❌ Нет {meta}. Загрузили train_dataset.zip?"

images = list((ROOT / "dataset" / "train").glob("*.jpg"))
images += list((ROOT / "dataset" / "train").glob("*.png"))

print(f"\n✓ OK! Структура готова:")
print(f"   ROOT: {ROOT}")
print(f"   Images: {len(images)}")
print(f"   metadata.jsonl: {meta.stat().st_size / 1024:.1f} KB")

# Проверка данных
import json
with open(meta) as f:
    records = [json.loads(line) for line in f]
print(f"   Records in metadata: {len(records)}")
print(f"   Sample fields: {len(json.loads(records[0]['ground_truth'])['gt_parse'])} полей")

In [ ]:
# --- Если использовали ОДИН архив flowvision_colab_all.zip ---
# Закомментируйте ячейку выше и раскомментируйте это:
#
# from google.colab import files
# import zipfile
# from pathlib import Path
# uploaded = files.upload()
# ROOT = Path("/content/FlowVision")
# ROOT.mkdir(parents=True, exist_ok=True)
# zipfile.ZipFile(list(uploaded.keys())[0]).extractall(ROOT)

## 2. Установка зависимостей

In [ ]:
import os
import sys

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

print("📦 Установка зависимостей...")
%pip install -q transformers pytorch-lightning sentencepiece timm albumentations accelerate
print("✓ OK")

## 3. Параметры обучения

In [ ]:
print("📋 Параметры обучения (для T4 GPU, 52 изображения):")
print()

MAX_EPOCHS = 25
LEARNING_RATE = 2e-5
ACCUMULATE_GRAD_BATCHES = 4
EARLY_STOP_PATIENCE = 5

config_info = {
    "MAX_EPOCHS": MAX_EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "ACCUMULATE_GRAD_BATCHES": ACCUMULATE_GRAD_BATCHES,
    "EARLY_STOP_PATIENCE": EARLY_STOP_PATIENCE,
    "batch_size": 1,
    "max_length": 512,
    "image_height": 960,
    "image_width": 1280,
    "use_fp16": True,
    "gradient_checkpointing": True,
}

for k, v in config_info.items():
    print(f"  {k}: {v}")

print("\n💡 Если OOM: --lowmem --freeze-encoder")
print("⚠️ НЕ используйте --minimal для реального обучения (max_length=128, заморожен encoder)")

## 4. Скачать базовую модель donut-base

In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel

os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

base_dir = ROOT / "models" / "donut-base"
if not (base_dir / "config.json").exists():
    print("📥 Скачивание базовой модели Akajackson/donut_rus...")
    base_dir.mkdir(parents=True, exist_ok=True)
    processor = DonutProcessor.from_pretrained("Akajackson/donut_rus")
    model = VisionEncoderDecoderModel.from_pretrained("Akajackson/donut_rus")
    processor.save_pretrained(base_dir)
    model.save_pretrained(base_dir)
    print(f"✓ Сохранена в {base_dir}")
    print(f"  Config size: {(base_dir / 'config.json').stat().st_size / 1024:.1f} KB")
else:
    print(f"✓ donut-base уже есть ({(base_dir / 'config.json').stat().st_size / 1024:.1f} KB)")

## Шаг 4: Обучение модели на T4 GPU

Режим `--lowmem`: оптимизирован для Tesla T4 16 GB
- Размер изображения: 1280×960
- FP16, gradient checkpointing, max_length: 512
- Сохраняется **лучший** checkpoint по `val_loss` (не последняя эпоха)
- `eos` = `</s_waybill>`, маска стартового `<s_waybill>` в loss

**Ожидаемое время на T4:** 30–90 минут

```bash
python train/train.py --lowmem          # обычно
python train/train.py --lowmem --freeze-encoder   # только при OOM
```

⚠️ **Не запускайте** `--minimal` — модель не научится полям.

In [ ]:
import os
import sys

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

MAX_EPOCHS = 25
EARLY_STOP_PATIENCE = 5  # при 50 эпохах можно чуть увеличить (в train.py по умолчанию 5)

print(f"🚀 Запуск обучения (--lowmem, max_epochs={MAX_EPOCHS})...\n")
print("   Должно появиться: «Сохраняем лучший checkpoint (val_loss): ...»\n")
!python train/train.py --lowmem --epochs {MAX_EPOCHS} --patience {EARLY_STOP_PATIENCE}

print("\n✓ Обучение завершено!")

cfg_final_dir = ROOT / "models" / "donut-trained-final"
print(f"   Модель: {cfg_final_dir}")
print(f"   Файлы: {[p.name for p in cfg_final_dir.glob('*') if p.is_file()]}")

# Кратко: val_loss по эпохам
from pathlib import Path
import pandas as pd

log_csvs = sorted((ROOT / "lightning_logs").glob("**/metrics.csv"))
if log_csvs:
    df = pd.read_csv(log_csvs[-1])
    tail = df[["epoch", "val_loss", "train_loss_epoch"]].dropna()
    if not tail.empty:
        print("\n📉 Метрики (последние строки):")
        print(tail.tail(8).to_string(index=False))
        last_val = tail["val_loss"].dropna().iloc[-1]
        print(f"\n   Последний val_loss: {last_val:.3f}  (цель ~2.5–4.0)")

## Шаг 5: Тест на одном изображении

Скачивайте zip **только если** в raw есть теги `<s_waybill_number>` или `<s_carrier_inn>` и `Полей` > 0.

In [ ]:
import json
import sys
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoTokenizer, DonutProcessor, VisionEncoderDecoderModel

sys.path.insert(0, str(ROOT))

from app.ocr.donut_format import TASK_END_TOKEN, apply_processor_image_size, resize_image_keep_aspect
from app.ocr.donut_inference import DonutInference

cfg_final_dir = ROOT / "models" / "donut-trained-final"
dataset_path = ROOT / "dataset" / "train"

with open(dataset_path / "metadata.jsonl", encoding="utf-8") as f:
    sample = json.loads(f.readline())

print(f"📷 Тест: {sample['file_name']}\n")

# 1) Токены в сохранённой модели
tok = AutoTokenizer.from_pretrained(cfg_final_dir)
unk = tok.unk_token_id
for t in ["<s_waybill>", TASK_END_TOKEN, "<s_waybill_number>"]:
    tid = tok.convert_tokens_to_ids(t)
    print(f"  {t}: {'OK' if tid != unk else 'UNK ❌'}")

# 2) Teacher-forcing loss на эталоне (должен быть < ~2.0)
proc = DonutProcessor.from_pretrained(cfg_final_dir)
model = VisionEncoderDecoderModel.from_pretrained(cfg_final_dir).to("cuda").eval()
rec = sample
img = Image.open(dataset_path / rec["file_name"]).convert("RGB")
apply_processor_image_size(proc, 960, 1280)
img_r = resize_image_keep_aspect(img, 1280, 960)
pv = proc(img_r, return_tensors="pt").pixel_values.to("cuda")
labels = proc.tokenizer(
    rec["target_sequence"],
    add_special_tokens=False,
    return_tensors="pt",
    max_length=512,
    truncation=True,
).input_ids.to("cuda")
labels[labels == proc.tokenizer.pad_token_id] = -100
with torch.no_grad():
    tf_loss = model(pixel_values=pv, labels=labels).loss.item()
print(f"\n  loss на разметке (teacher forcing): {tf_loss:.3f}  (<2.0 — хорошо)")

del model, proc, pv, labels
torch.cuda.empty_cache()

# 3) Инференс (DonutInference: eos=</s_waybill>, beams=4, anti-repeat)
engine = DonutInference(
    model_path=cfg_final_dir,
    device="cuda",
    max_length=512,
    image_width=1280,
    image_height=960,
    num_beams=4,
    repetition_penalty=1.15,
)

gt_parse, seq = engine.predict_image(img)
has_field_tags = "<s_waybill_number>" in seq or "<s_carrier_inn>" in seq

print(f"\n✓ Предсказание: {len(gt_parse)} полей")
if not gt_parse:
    print("   ⚠️ ПУСТО — дотренируйте или проверьте val_loss в шаге 4")
else:
    for field, value in gt_parse.items():
        print(f"   {field}: {value}")

print(f"\n🔍 Теги полей в raw: {'✓' if has_field_tags else '✗ нет — не скачивайте zip'}")
print(f"   {repr(seq[:500])}")

print(f"\n✓ Эталон ({len(json.loads(sample['ground_truth'])['gt_parse'])} полей), первые 5:")
for field, value in list(json.loads(sample["ground_truth"])["gt_parse"].items())[:5]:
    print(f"   {field}: {value}")

## Шаг 6: Скачать обученную модель на ПК

Распакуйте `donut-trained-final.zip` в:
```
d:\GitHub\FlowVision\models\donut-trained-final\
```

Там должны быть файлы:
- `config.json` (в т.ч. `eos_token_id` для `</s_waybill>`)
- `model.safetensors`
- `tokenizer.json`
- `preprocessor_config.json`

Перед скачиванием в шаге 5 должны быть **теги полей** в raw и **Полей > 0**.

In [ ]:
import shutil
from google.colab import files

print("📦 Упаковка модели...")
zip_out = "/content/donut-trained-final.zip"
shutil.make_archive("/content/donut-trained-final", "zip", str(cfg_final_dir))
zip_size = Path(zip_out).stat().st_size / 1024 / 1024
print(f"✓ Создан архив: {zip_size:.1f} MB\n")

print("⬇️  Скачивание на ПК...")
files.download(zip_out)

## Шаг 7 (опционально): Сохранить модель на Google Drive

Если сессия Colab разорвётся, модель останется на Drive и не потребуется переобучение.

⚠️ Требует авторизации в Google Drive!

In [ ]:
from google.colab import drive
import shutil
from pathlib import Path

print("🔐 Авторизация в Google Drive...")
drive.mount("/content/drive")

dest = Path("/content/drive/MyDrive/FlowVision_models/donut-trained-final")
print(f"💾 Сохранение в Drive: {dest}")

if dest.exists():
    shutil.rmtree(dest)
    
shutil.copytree(cfg_final_dir, dest)
print(f"✓ Модель сохранена на Drive!")